In [ ]:
!pip install -q cma

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    roc_curve, auc, classification_report
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

import cma

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/kaggle/input/liver-histopathology"  # flat class-folder dataset: F0, F1, F2, F3, F4
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 5  # F0, F1, F2, F3, F4 (liver fibrosis stages)
CLASS_NAMES = None  # filled after loading dataset


In [ ]:

from sklearn.model_selection import train_test_split

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Dataset is flat class folders (F0..F4) -> no pre-made Training/Testing split,
# so we load once with ImageFolder and split 70/15/15 ourselves (stratified by class via indices).
full_dataset = datasets.ImageFolder(DATA_DIR, transform=eval_transform)
CLASS_NAMES = full_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

targets = np.array(full_dataset.targets)
train_idx, temp_idx = train_test_split(
    np.arange(len(targets)), test_size=0.30, stratify=targets, random_state=SEED
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=targets[temp_idx], random_state=SEED
)

train_dataset = torch.utils.data.Subset(
    datasets.ImageFolder(DATA_DIR, transform=train_transform), train_idx
)
val_dataset = torch.utils.data.Subset(full_dataset, val_idx)
test_dataset = torch.utils.data.Subset(full_dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Classes: {CLASS_NAMES}")
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")


# --- Class weights for imbalance handling (F0..F4 liver fibrosis stages) ---
def compute_class_weights(target_array, num_classes):
    counts = np.bincount(target_array, minlength=num_classes)
    counts = np.maximum(counts, 1)
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)

CLASS_WEIGHTS = compute_class_weights(targets[train_idx], NUM_CLASSES).to(DEVICE)
print("Class distribution (train):", np.bincount(targets[train_idx], minlength=NUM_CLASSES))
print("Class weights (inverse frequency):", CLASS_WEIGHTS.cpu().numpy())


In [ ]:
ORDINAL_LAMBDA = 0.3   # weight of the ordinal-distance penalty
FOCAL_GAMMA = 2.0      # focusing parameter for focal loss


class FocalLoss(nn.Module):
    """Cross-entropy with class weights (imbalance) + a focusing term (hard examples)."""
    def __init__(self, class_weights=None, gamma=2.0):
        super().__init__()
        self.class_weights = class_weights
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.class_weights, reduction="none")
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()


def ordinal_distance_penalty(logits, targets, num_classes):
    """F0..F4 are ordered stages, not independent categories. This penalizes
    predictions that are ordinally far from the true stage (e.g. predicting F4
    when the truth is F0 is worse than predicting F1), on top of plain CE which
    treats every wrong class equally."""
    probs = F.softmax(logits, dim=1)
    class_indices = torch.arange(num_classes, device=logits.device).float()
    expected_idx = (probs * class_indices).sum(dim=1)
    true_idx = targets.float()
    return F.mse_loss(expected_idx, true_idx)


focal_loss_fn = FocalLoss(class_weights=CLASS_WEIGHTS, gamma=FOCAL_GAMMA)


def domain_loss(logits, targets, num_classes):
    return focal_loss_fn(logits, targets) + ORDINAL_LAMBDA * ordinal_distance_penalty(logits, targets, num_classes)


In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        y = torch.cat([avg_out, max_out], dim=1)
        y = self.conv(y)
        return self.sigmoid(y)


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x

In [ ]:
def build_teacher(name, num_classes):
    if name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    else:
        raise ValueError(f"Unknown teacher {name}")
    return model


def train_one_teacher(model, train_loader, val_loader, epochs=5, lr=1e-4):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * x.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                out = model(x)
                preds = out.argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        val_acc = correct / total
        print(f"[Teacher {model.__class__.__name__}] Epoch {epoch+1}/{epochs} "
              f"train_loss={train_loss:.4f} val_acc={val_acc:.4f}")
    return model


class TeacherEnsemble(nn.Module):
    """Wraps multiple trained teachers, returns averaged softmax probabilities."""
    def __init__(self, teachers, weights=None):
        super().__init__()
        self.teachers = nn.ModuleList(teachers)
        for t in self.teachers:
            t.eval()
            for p in t.parameters():
                p.requires_grad = False
        n = len(teachers)
        self.weights = weights if weights is not None else [1.0 / n] * n

    @torch.no_grad()
    def forward(self, x):
        probs = 0
        for w, t in zip(self.weights, self.teachers):
            logits = t(x)
            probs = probs + w * F.softmax(logits, dim=1)
        return probs  # already probabilities, summed to ~1

In [ ]:
class StudentModel(nn.Module):
    def __init__(self, num_classes, use_cbam=True):
        super().__init__()
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = backbone.features
        out_channels = 1280  # efficientnet_b0 final feature channels

        self.use_cbam = use_cbam
        if use_cbam:
            self.attention = CBAM(out_channels)
        else:
            self.attention = SEBlock(out_channels)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(out_channels, num_classes)

    def forward(self, x, return_features=False):
        x = self.features(x)
        x = self.attention(x)
        x = self.pool(x)
        feat = torch.flatten(x, 1)
        out = self.classifier(self.dropout(feat))
        if return_features:
            return out, feat
        return out

In [ ]:
def distillation_loss(student_logits, teacher_probs, true_labels, alpha, temperature):
    ce_loss = domain_loss(student_logits, true_labels, NUM_CLASSES)  # focal + ordinal, replaces plain CE

    student_log_soft = F.log_softmax(student_logits / temperature, dim=1)
    teacher_soft = teacher_probs  # already softmax probabilities from ensemble
    # sharpen/soften teacher probs with temperature by re-deriving via log then softmax
    teacher_soft_T = F.softmax(torch.log(teacher_soft + 1e-8) / temperature, dim=1)

    kd_loss = F.kl_div(student_log_soft, teacher_soft_T, reduction="batchmean") * (temperature ** 2)

    return alpha * kd_loss + (1 - alpha) * ce_loss

In [ ]:
def train_student(student, teacher_ensemble, train_loader, val_loader,
                   alpha, temperature, lr, epochs, verbose=True):
    student = student.to(DEVICE)
    optimizer = torch.optim.Adam(student.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        student.train()
        running_loss, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            student_logits = student(x)
            with torch.no_grad():
                teacher_probs = teacher_ensemble(x)
            loss = distillation_loss(student_logits, teacher_probs, y, alpha, temperature)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)
            preds = student_logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = correct / total

        student.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                student_logits = student(x)
                teacher_probs = teacher_ensemble(x)
                loss = distillation_loss(student_logits, teacher_probs, y, alpha, temperature)
                val_running_loss += loss.item() * x.size(0)
                preds = student_logits.argmax(dim=1)
                val_correct += (preds == y).sum().item()
                val_total += y.size(0)

        val_loss = val_running_loss / len(val_loader.dataset)
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if verbose:
            print(f"Epoch {epoch+1}/{epochs} | train_loss={train_loss:.4f} "
                  f"train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    return student, history

In [ ]:
def cma_es_objective(params, teacher_ensemble, train_loader, val_loader):
    """params = [alpha_raw, temperature_raw, lr_raw] in unconstrained space,
    mapped into valid ranges."""
    alpha = 1 / (1 + np.exp(-params[0]))            # sigmoid -> (0,1)
    temperature = 1.0 + 9.0 * (1 / (1 + np.exp(-params[1])))  # -> (1,10)
    lr = 10 ** (-5 + 3 * (1 / (1 + np.exp(-params[2]))))      # -> (1e-5, 1e-2)

    student = StudentModel(NUM_CLASSES, use_cbam=True)
    _, history = train_student(
        student, teacher_ensemble, train_loader, val_loader,
        alpha=alpha, temperature=temperature, lr=lr, epochs=2, verbose=False
    )
    val_loss = history["val_loss"][-1]
    print(f"  tried alpha={alpha:.3f} temp={temperature:.3f} lr={lr:.2e} -> val_loss={val_loss:.4f}")
    return val_loss


def run_cma_es_search(teacher_ensemble, train_loader, val_loader, max_iters=8, popsize=4):
    x0 = [0.0, 0.0, 0.0]
    sigma0 = 0.8
    es = cma.CMAEvolutionStrategy(x0, sigma0, {"popsize": popsize, "maxiter": max_iters, "verbose": -9})

    while not es.stop():
        solutions = es.ask()
        fitnesses = [
            cma_es_objective(sol, teacher_ensemble, train_loader, val_loader)
            for sol in solutions
        ]
        es.tell(solutions, fitnesses)
        es.disp()

    best = es.result.xbest
    best_alpha = 1 / (1 + np.exp(-best[0]))
    best_temp = 1.0 + 9.0 * (1 / (1 + np.exp(-best[1])))
    best_lr = 10 ** (-5 + 3 * (1 / (1 + np.exp(-best[2]))))
    print(f"Best found -> alpha={best_alpha:.4f}, temperature={best_temp:.4f}, lr={best_lr:.6f}")
    return best_alpha, best_temp, best_lr

In [ ]:
teacher1 = build_teacher("resnet50", NUM_CLASSES)
teacher1 = train_one_teacher(teacher1, train_loader, val_loader, epochs=5, lr=1e-4)

teacher2 = build_teacher("densenet121", NUM_CLASSES)
teacher2 = train_one_teacher(teacher2, train_loader, val_loader, epochs=5, lr=1e-4)

teacher_ensemble = TeacherEnsemble([teacher1, teacher2]).to(DEVICE)

In [ ]:
best_alpha, best_temp, best_lr = run_cma_es_search(
    teacher_ensemble, train_loader, val_loader, max_iters=6, popsize=4
)

final_student = StudentModel(NUM_CLASSES, use_cbam=True)
final_student, history = train_student(
    final_student, teacher_ensemble, train_loader, val_loader,
    alpha=best_alpha, temperature=best_temp, lr=best_lr, epochs=20, verbose=True
)

torch.save(final_student.state_dict(), "/kaggle/working/student_model.pth")

In [ ]:
final_student.eval()
all_preds, all_labels, all_probs, all_feats = [], [], [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        logits, feats = final_student(x, return_features=True)
        probs = F.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())
        all_probs.extend(probs.cpu().numpy())
        all_feats.extend(feats.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)
all_feats = np.array(all_feats)

In [ ]:
acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, support = precision_recall_fscore_support(
    all_labels, all_preds, average=None, labels=range(NUM_CLASSES)
)
precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
    all_labels, all_preds, average="weighted"
)

print(f"Overall Accuracy: {acc:.4f}")
print(f"Weighted Precision: {precision_w:.4f} | Weighted Recall: {recall_w:.4f} | Weighted F1: {f1_w:.4f}\n")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

metrics_df = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1,
    "Support": support
})
print(metrics_df)

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150)
plt.show()

FP = cm.sum(axis=0) - np.diag(cm)
FN = cm.sum(axis=1) - np.diag(cm)
TP = np.diag(cm)
TN = cm.sum() - (FP + FN + TP)

TPR = TP / (TP + FN)        # Sensitivity / Recall
TNR = TN / (TN + FP)        # Specificity
FPR = FP / (FP + TN)
FNR = FN / (FN + TP)

rates_df = pd.DataFrame({
    "Class": CLASS_NAMES,
    "TPR (Sensitivity)": TPR,
    "TNR (Specificity)": TNR,
    "FPR": FPR,
    "FNR": FNR
})
print(rates_df)

In [ ]:
labels_bin = label_binarize(all_labels, classes=range(NUM_CLASSES))

plt.figure(figsize=(7, 6))
for i in range(NUM_CLASSES):
    fpr_i, tpr_i, _ = roc_curve(labels_bin[:, i], all_probs[:, i])
    roc_auc_i = auc(fpr_i, tpr_i)
    plt.plot(fpr_i, tpr_i, label=f"{CLASS_NAMES[i]} (AUC = {roc_auc_i:.3f})")

plt.plot([0, 1], [0, 1], "k--", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (One-vs-Rest)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("/kaggle/working/roc_curves.png", dpi=150)
plt.show()

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], label="Train Loss")
plt.plot(epochs_range, history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["train_acc"], label="Train Accuracy")
plt.plot(epochs_range, history["val_acc"], label="Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy Curve")
plt.legend()

plt.tight_layout()
plt.savefig("/kaggle/working/loss_accuracy_curves.png", dpi=150)
plt.show()

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, init="pca")
feats_2d = tsne.fit_transform(all_feats)

plt.figure(figsize=(7, 6))
for i in range(NUM_CLASSES):
    idx = all_labels == i
    plt.scatter(feats_2d[idx, 0], feats_2d[idx, 1], label=CLASS_NAMES[i], s=10, alpha=0.7)
plt.legend()
plt.title("t-SNE of Student Model Features (Test Set)")
plt.tight_layout()
plt.savefig("/kaggle/working/tsne.png", dpi=150)
plt.show()

In [ ]:
!pip install -q cma thop

import os
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    roc_curve, auc, classification_report
)
from sklearn.preprocessing import label_binarize

import cma
from thop import profile


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/kaggle/input/datasets/sabbir4724/liver-histopathology"          # flat class folders: F0..F4
CHECKPOINT_DIR = "/kaggle/working/checkpoints"            # per-fold resumable checkpoints
RESULTS_DIR = "/kaggle/working/kfold_results"             # aggregated CV results
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
N_FOLDS = 5

TEACHER_EPOCHS = 5
TEACHER_LR = 1e-4
CMA_MAX_ITERS = 6          # lower this (e.g. 3-4) if 5-fold total time is too much for your Kaggle quota
CMA_POPSIZE = 4
CMA_TRIAL_EPOCHS = 2
STUDENT_EPOCHS = 20

ORDINAL_LAMBDA = 0.3
FOCAL_GAMMA = 2.0

print(f"Device: {DEVICE}")


train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Two ImageFolder views of the SAME files (same order), one per transform pipeline.
base_dataset_train = datasets.ImageFolder(DATA_DIR, transform=train_transform)
base_dataset_eval = datasets.ImageFolder(DATA_DIR, transform=eval_transform)

CLASS_NAMES = base_dataset_eval.classes
NUM_CLASSES = len(CLASS_NAMES)
targets = np.array(base_dataset_eval.targets)

print(f"Classes: {CLASS_NAMES}")
print(f"Total samples: {len(targets)}")
print("Class distribution (full dataset):", np.bincount(targets, minlength=NUM_CLASSES))


def compute_class_weights(target_array, num_classes):
    counts = np.bincount(target_array, minlength=num_classes)
    counts = np.maximum(counts, 1)
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)


class FocalLoss(nn.Module):
    def __init__(self, class_weights=None, gamma=2.0):
        super().__init__()
        self.class_weights = class_weights
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.class_weights, reduction="none")
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()


def ordinal_distance_penalty(logits, targets, num_classes):
    probs = F.softmax(logits, dim=1)
    class_indices = torch.arange(num_classes, device=logits.device).float()
    expected_idx = (probs * class_indices).sum(dim=1)
    true_idx = targets.float()
    return F.mse_loss(expected_idx, true_idx)


def make_domain_loss(class_weights):
    focal_fn = FocalLoss(class_weights=class_weights, gamma=FOCAL_GAMMA)
    def domain_loss(logits, targets, num_classes):
        return focal_fn(logits, targets) + ORDINAL_LAMBDA * ordinal_distance_penalty(logits, targets, num_classes)
    return domain_loss


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        y = torch.cat([avg_out, max_out], dim=1)
        y = self.conv(y)
        return self.sigmoid(y)


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x


def build_teacher(name, num_classes):
    if name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == "densenet121":
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    else:
        raise ValueError(f"Unknown teacher {name}")
    return model


def train_one_teacher(model, train_loader, val_loader, epochs=5, lr=1e-4):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * x.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                out = model(x)
                preds = out.argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        val_acc = correct / total
        print(f"    [Teacher {model.__class__.__name__}] Epoch {epoch+1}/{epochs} "
              f"train_loss={train_loss:.4f} val_acc={val_acc:.4f}")
    return model


class TeacherEnsemble(nn.Module):
    def __init__(self, teachers, weights=None):
        super().__init__()
        self.teachers = nn.ModuleList(teachers)
        for t in self.teachers:
            t.eval()
            for p in t.parameters():
                p.requires_grad = False
        n = len(teachers)
        self.weights = weights if weights is not None else [1.0 / n] * n

    @torch.no_grad()
    def forward(self, x):
        probs = 0
        for w, t in zip(self.weights, self.teachers):
            logits = t(x)
            probs = probs + w * F.softmax(logits, dim=1)
        return probs


class StudentModel(nn.Module):
    def __init__(self, num_classes, use_cbam=True):
        super().__init__()
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = backbone.features
        out_channels = 1280

        self.use_cbam = use_cbam
        if use_cbam:
            self.attention = CBAM(out_channels)
        else:
            self.attention = SEBlock(out_channels)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(out_channels, num_classes)

    def forward(self, x, return_features=False):
        x = self.features(x)
        x = self.attention(x)
        x = self.pool(x)
        feat = torch.flatten(x, 1)
        out = self.classifier(self.dropout(feat))
        if return_features:
            return out, feat
        return out


def distillation_loss(domain_loss_fn, student_logits, teacher_probs, true_labels, alpha, temperature, num_classes):
    ce_loss = domain_loss_fn(student_logits, true_labels, num_classes)

    student_log_soft = F.log_softmax(student_logits / temperature, dim=1)
    teacher_soft = teacher_probs
    teacher_soft_T = F.softmax(torch.log(teacher_soft + 1e-8) / temperature, dim=1)

    kd_loss = F.kl_div(student_log_soft, teacher_soft_T, reduction="batchmean") * (temperature ** 2)
    return alpha * kd_loss + (1 - alpha) * ce_loss


def train_student(student, teacher_ensemble, train_loader, val_loader, domain_loss_fn, num_classes,
                   alpha, temperature, lr, epochs, verbose=True):
    student = student.to(DEVICE)
    optimizer = torch.optim.Adam(student.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        student.train()
        running_loss, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            student_logits = student(x)
            with torch.no_grad():
                teacher_probs = teacher_ensemble(x)
            loss = distillation_loss(domain_loss_fn, student_logits, teacher_probs, y, alpha, temperature, num_classes)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)
            preds = student_logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = correct / total

        student.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                student_logits = student(x)
                teacher_probs = teacher_ensemble(x)
                loss = distillation_loss(domain_loss_fn, student_logits, teacher_probs, y, alpha, temperature, num_classes)
                val_running_loss += loss.item() * x.size(0)
                preds = student_logits.argmax(dim=1)
                val_correct += (preds == y).sum().item()
                val_total += y.size(0)

        val_loss = val_running_loss / len(val_loader.dataset)
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if verbose:
            print(f"    Epoch {epoch+1}/{epochs} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
                  f"| val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    return student, history


def cma_es_objective(params, teacher_ensemble, train_loader, val_loader, domain_loss_fn, num_classes):
    alpha = 1 / (1 + np.exp(-params[0]))
    temperature = 1.0 + 9.0 * (1 / (1 + np.exp(-params[1])))
    lr = 10 ** (-5 + 3 * (1 / (1 + np.exp(-params[2]))))

    student = StudentModel(num_classes, use_cbam=True)
    _, history = train_student(
        student, teacher_ensemble, train_loader, val_loader, domain_loss_fn, num_classes,
        alpha=alpha, temperature=temperature, lr=lr, epochs=CMA_TRIAL_EPOCHS, verbose=False
    )
    val_loss = history["val_loss"][-1]
    print(f"      tried alpha={alpha:.3f} temp={temperature:.3f} lr={lr:.2e} -> val_loss={val_loss:.4f}")
    return val_loss


def run_cma_es_search(teacher_ensemble, train_loader, val_loader, domain_loss_fn, num_classes,
                       max_iters=CMA_MAX_ITERS, popsize=CMA_POPSIZE):
    x0 = [0.0, 0.0, 0.0]
    sigma0 = 0.8
    es = cma.CMAEvolutionStrategy(x0, sigma0, {"popsize": popsize, "maxiter": max_iters, "verbose": -9})

    while not es.stop():
        solutions = es.ask()
        fitnesses = [
            cma_es_objective(sol, teacher_ensemble, train_loader, val_loader, domain_loss_fn, num_classes)
            for sol in solutions
        ]
        es.tell(solutions, fitnesses)
        es.disp()

    best = es.result.xbest
    best_alpha = 1 / (1 + np.exp(-best[0]))
    best_temp = 1.0 + 9.0 * (1 / (1 + np.exp(-best[1])))
    best_lr = 10 ** (-5 + 3 * (1 / (1 + np.exp(-best[2]))))
    print(f"    Best -> alpha={best_alpha:.4f}, temperature={best_temp:.4f}, lr={best_lr:.6f}")
    return best_alpha, best_temp, best_lr


def fold_dir(fold_idx):
    d = os.path.join(CHECKPOINT_DIR, f"fold_{fold_idx}")
    os.makedirs(d, exist_ok=True)
    return d


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def load_json(path):
    with open(path, "r") as f:
        return json.load(f)


skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_indices = np.arange(len(targets))

results_csv_path = os.path.join(RESULTS_DIR, "kfold_metrics.csv")

if os.path.exists(results_csv_path):
    fold_results = pd.read_csv(results_csv_path).to_dict("records")
    done_folds = set(int(r["fold"]) for r in fold_results)
    print(f"Resuming: {len(done_folds)} fold(s) already completed -> {sorted(done_folds)}")
else:
    fold_results = []
    done_folds = set()

for fold_idx, (trainval_idx, test_idx) in enumerate(skf.split(all_indices, targets)):
    fold_num = fold_idx + 1
    print(f"\n{'='*70}\nFOLD {fold_num}/{N_FOLDS}\n{'='*70}")

    if fold_num in done_folds:
        print(f"  Fold {fold_num} already completed -> skipping.")
        continue

    fdir = fold_dir(fold_num)

    tv_targets = targets[trainval_idx]
    inner_train_idx, inner_val_idx = train_test_split(
        trainval_idx, test_size=0.15, stratify=tv_targets, random_state=SEED
    )

    train_dataset = Subset(base_dataset_train, inner_train_idx)
    val_dataset = Subset(base_dataset_eval, inner_val_idx)
    test_dataset = Subset(base_dataset_eval, test_idx)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    class_weights = compute_class_weights(targets[inner_train_idx], NUM_CLASSES).to(DEVICE)
    domain_loss_fn = make_domain_loss(class_weights)
    print(f"  train={len(train_dataset)}  val={len(val_dataset)}  test={len(test_dataset)}")

    t1_path = os.path.join(fdir, "teacher_resnet50.pth")
    t2_path = os.path.join(fdir, "teacher_densenet121.pth")

    teacher1 = build_teacher("resnet50", NUM_CLASSES).to(DEVICE)
    if os.path.exists(t1_path):
        teacher1.load_state_dict(torch.load(t1_path, map_location=DEVICE))
        print("  Loaded cached teacher1 (resnet50)")
    else:
        teacher1 = train_one_teacher(teacher1, train_loader, val_loader, epochs=TEACHER_EPOCHS, lr=TEACHER_LR)
        torch.save(teacher1.state_dict(), t1_path)

    teacher2 = build_teacher("densenet121", NUM_CLASSES).to(DEVICE)
    if os.path.exists(t2_path):
        teacher2.load_state_dict(torch.load(t2_path, map_location=DEVICE))
        print("  Loaded cached teacher2 (densenet121)")
    else:
        teacher2 = train_one_teacher(teacher2, train_loader, val_loader, epochs=TEACHER_EPOCHS, lr=TEACHER_LR)
        torch.save(teacher2.state_dict(), t2_path)

    teacher_ensemble = TeacherEnsemble([teacher1, teacher2]).to(DEVICE)

    params_path = os.path.join(fdir, "best_params.json")
    if os.path.exists(params_path):
        best_params = load_json(params_path)
        best_alpha, best_temp, best_lr = best_params["alpha"], best_params["temperature"], best_params["lr"]
        print(f"  Loaded cached best params: alpha={best_alpha:.4f} temp={best_temp:.4f} lr={best_lr:.6f}")
    else:
        best_alpha, best_temp, best_lr = run_cma_es_search(
            teacher_ensemble, train_loader, val_loader, domain_loss_fn, NUM_CLASSES,
            max_iters=CMA_MAX_ITERS, popsize=CMA_POPSIZE
        )
        save_json({"alpha": best_alpha, "temperature": best_temp, "lr": best_lr}, params_path)

    student_path = os.path.join(fdir, "student_model.pth")
    history_path = os.path.join(fdir, "history.json")
    final_student = StudentModel(NUM_CLASSES, use_cbam=True).to(DEVICE)
    if os.path.exists(student_path):
        final_student.load_state_dict(torch.load(student_path, map_location=DEVICE))
        print("  Loaded cached final student")
    else:
        final_student, history = train_student(
            final_student, teacher_ensemble, train_loader, val_loader, domain_loss_fn, NUM_CLASSES,
            alpha=best_alpha, temperature=best_temp, lr=best_lr, epochs=STUDENT_EPOCHS, verbose=True
        )
        torch.save(final_student.state_dict(), student_path)
        save_json(history, history_path)

    final_student.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE)
            logits = final_student(x)
            probs = F.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    acc = accuracy_score(all_labels, all_preds)
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted")
    precision_pc, recall_pc, f1_pc, support_pc = precision_recall_fscore_support(
        all_labels, all_preds, average=None, labels=range(NUM_CLASSES)
    )

    labels_bin = label_binarize(all_labels, classes=range(NUM_CLASSES))
    auc_scores = []
    for c in range(NUM_CLASSES):
        fpr_c, tpr_c, _ = roc_curve(labels_bin[:, c], all_probs[:, c])
        auc_scores.append(auc(fpr_c, tpr_c))
    macro_auc = float(np.mean(auc_scores))

    cm = confusion_matrix(all_labels, all_preds, labels=range(NUM_CLASSES))
    np.save(os.path.join(fdir, "confusion_matrix.npy"), cm)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Fold {fold_num} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(fdir, "confusion_matrix.png"), dpi=600)
    plt.close()

    fold_row = {
        "fold": fold_num,
        "accuracy": acc,
        "precision_weighted": precision_w,
        "recall_weighted": recall_w,
        "f1_weighted": f1_w,
        "macro_auc": macro_auc,
        "best_alpha": best_alpha,
        "best_temperature": best_temp,
        "best_lr": best_lr,
    }
    for c, cname in enumerate(CLASS_NAMES):
        fold_row[f"precision_{cname}"] = precision_pc[c]
        fold_row[f"recall_{cname}"] = recall_pc[c]
        fold_row[f"f1_{cname}"] = f1_pc[c]

    fold_results.append(fold_row)
    pd.DataFrame(fold_results).to_csv(results_csv_path, index=False)
    print(f"  Fold {fold_num} done -> acc={acc:.4f}  weighted_F1={f1_w:.4f}  macro_AUC={macro_auc:.4f}")

print("\nAll folds complete.")


results_df = pd.DataFrame(fold_results).sort_values("fold").reset_index(drop=True)
print(results_df)

summary_cols = ["accuracy", "precision_weighted", "recall_weighted", "f1_weighted", "macro_auc"]
summary = results_df[summary_cols].agg(["mean", "std"]).T
summary.columns = ["mean", "std"]

print(f"\n=== {N_FOLDS}-Fold Cross-Validation Summary ===")
print(summary)

summary.to_csv(os.path.join(RESULTS_DIR, "kfold_summary.csv"))
results_df.to_csv(os.path.join(RESULTS_DIR, "kfold_metrics_final.csv"), index=False)

plt.figure(figsize=(7, 5))
results_df[["accuracy", "f1_weighted", "macro_auc"]].boxplot()
plt.title(f"{N_FOLDS}-Fold CV Metric Spread")
plt.ylabel("Score")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "kfold_boxplot.png"), dpi=600)
plt.show()

agg_cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
for fold_num in range(1, N_FOLDS + 1):
    cm_path = os.path.join(fold_dir(fold_num), "confusion_matrix.npy")
    if os.path.exists(cm_path):
        agg_cm += np.load(cm_path)

plt.figure(figsize=(6, 5))
sns.heatmap(agg_cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Aggregated Confusion Matrix ({N_FOLDS}-Fold)")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "aggregated_confusion_matrix.png"), dpi=600)
plt.show()


table_rows = []
for _, row in results_df.iterrows():
    for j, cname in enumerate(CLASS_NAMES):
        table_rows.append({
            "Fold": f"Fold {int(row['fold'])}" if j == 0 else "",
            "Class": cname,
            "Precision (%)": round(row[f"precision_{cname}"] * 100, 1),
            "Recall (%)": round(row[f"recall_{cname}"] * 100, 1),
            "F1-score (%)": round(row[f"f1_{cname}"] * 100, 1),
            "Accuracy (%)": round(row["accuracy"] * 100, 1),
        })

report_df = pd.DataFrame(table_rows)

mean_std_rows = []
for j, cname in enumerate(CLASS_NAMES):
    p = results_df[f"precision_{cname}"] * 100
    r = results_df[f"recall_{cname}"] * 100
    f = results_df[f"f1_{cname}"] * 100
    a = results_df["accuracy"] * 100
    mean_std_rows.append({
        "Fold": "Mean ± Std" if j == 0 else "",
        "Class": cname,
        "Precision (%)": f"{p.mean():.1f} ± {p.std():.2f}",
        "Recall (%)": f"{r.mean():.1f} ± {r.std():.2f}",
        "F1-score (%)": f"{f.mean():.1f} ± {f.std():.2f}",
        "Accuracy (%)": f"{a.mean():.2f} ± {a.std():.2f}",
    })
mean_std_df = pd.DataFrame(mean_std_rows)

final_report_df = pd.concat([report_df, mean_std_df], ignore_index=True)
final_report_df.to_csv(os.path.join(RESULTS_DIR, "kfold_report_table.csv"), index=False)
final_report_df


n_rows = len(final_report_df)
n_cols = len(final_report_df.columns)
row_h = 0.6

fig, ax = plt.subplots(figsize=(11, 0.6 + row_h * (n_rows + 1)))
ax.axis("off")

tbl = ax.table(
    cellText=final_report_df.values,
    colLabels=final_report_df.columns,
    cellLoc="left",
    colLoc="left",
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1, 2.0)

n_classes = len(CLASS_NAMES)
mean_std_start_row = n_rows - n_classes + 1

for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("white")
    cell.set_linewidth(0)
    if r == 0:
        cell.set_text_props(weight="bold")
        cell.visible_edges = "B"
        cell.set_edgecolor("black")
        cell.set_linewidth(1.2)
    elif r == mean_std_start_row:
        cell.visible_edges = "T"
        cell.set_edgecolor("black")
        cell.set_linewidth(1.2)
        if c <= 1:
            cell.set_text_props(weight="bold")

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "kfold_report_table.png"), dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
N_FOLDS = 5

# Load all histories
histories = {}
for fold in range(1, N_FOLDS + 1):
    path = os.path.join(CHECKPOINT_DIR, f"fold_{fold}", "history.json")
    if os.path.exists(path):
        with open(path) as f:
            histories[fold] = json.load(f)
        print(f"Fold {fold}: {len(histories[fold]['train_acc'])} epochs")
    else:
        print(f"Missing: fold {fold}")


colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# ====================== Accuracy ======================
ax = axes[0]
for i, fold in enumerate(sorted(histories.keys())):
    h = histories[fold]
    epochs = np.arange(1, len(h["train_acc"]) + 1)
    
    ax.plot(epochs, h["train_acc"], color=colors[i], linestyle="-", 
            linewidth=2.0, label=f"Train Fold {fold}")
    ax.plot(epochs, h["val_acc"], color=colors[i], linestyle="--", 
            linewidth=2.0, label=f"Val Fold {fold}")

ax.set_title("Accuracy Across Folds", fontsize=14, fontweight="bold")
ax.set_xlabel("Epochs", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_ylim(0.55, 0.88)
ax.legend(fontsize=8, loc="lower right", ncol=2, framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_axisbelow(True)

# ====================== Loss ======================
ax = axes[1]
for i, fold in enumerate(sorted(histories.keys())):
    h = histories[fold]
    epochs = np.arange(1, len(h["train_loss"]) + 1)
    
    ax.plot(epochs, h["train_loss"], color=colors[i], linestyle="-", 
            linewidth=2.0, label=f"Train Fold {fold}")
    ax.plot(epochs, h["val_loss"], color=colors[i], linestyle="--", 
            linewidth=2.0, label=f"Val Fold {fold}")

ax.set_title("Loss Across Folds", fontsize=14, fontweight="bold")
ax.set_xlabel("Epochs", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.legend(fontsize=8, loc="upper right", ncol=2, framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("/kaggle/working/kfold_results/accuracy_loss_across_folds_fixed.png", 
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
from itertools import cycle

# ====================== Config ======================
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
RESULTS_DIR = "/kaggle/working/kfold_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_FOLDS = 5
NUM_CLASSES = len(CLASS_NAMES)          
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

# ====================== Re-evaluate all folds & save probs ======================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_indices = np.arange(len(targets))

fold_probs = []      # list of (n_samples, n_classes)
fold_labels = []     # list of (n_samples,)

for fold_idx, (trainval_idx, test_idx) in enumerate(skf.split(all_indices, targets)):
    fold_num = fold_idx + 1
    print(f"Evaluating Fold {fold_num}...")

    test_dataset = Subset(base_dataset_eval, test_idx)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    student = StudentModel(NUM_CLASSES, use_cbam=True).to(DEVICE)
    student_path = os.path.join(CHECKPOINT_DIR, f"fold_{fold_num}", "student_model.pth")
    student.load_state_dict(torch.load(student_path, map_location=DEVICE))
    student.eval()

    probs_list, labels_list = [], []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE)
            logits = student(x)
            probs = F.softmax(logits, dim=1)
            probs_list.append(probs.cpu().numpy())
            labels_list.append(y.numpy())

    fold_probs.append(np.concatenate(probs_list, axis=0))
    fold_labels.append(np.concatenate(labels_list, axis=0))

    
    np.save(os.path.join(CHECKPOINT_DIR, f"fold_{fold_num}", "test_probs.npy"), fold_probs[-1])
    np.save(os.path.join(CHECKPOINT_DIR, f"fold_{fold_num}", "test_labels.npy"), fold_labels[-1])

print("All folds evaluated & saved.\n")

# ====================== Plot Mean ROC Curve ======================
# Macro-average ROC (One-vs-Rest)

plt.figure(figsize=(8, 7))


mean_fpr = np.linspace(0, 1, 100)
all_tprs = {c: [] for c in range(NUM_CLASSES)}
all_aucs = {c: [] for c in range(NUM_CLASSES)}

colors = cycle(["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"])

for c, color in zip(range(NUM_CLASSES), colors):
    tprs = []
    aucs = []
    
    for fold in range(N_FOLDS):
        y_true = fold_labels[fold]
        y_prob = fold_probs[fold][:, c]
        
        # Binary label for this class
        y_true_bin = (y_true == c).astype(int)
        
        fpr, tpr, _ = roc_curve(y_true_bin, y_prob)
        roc_auc = auc(fpr, tpr)
        
        # Interpolate
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        aucs.append(roc_auc)
    
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = np.mean(aucs)
    std_auc = np.std(aucs)
    
    all_tprs[c] = mean_tpr
    all_aucs[c] = (mean_auc, std_auc)
    
    plt.plot(mean_fpr, mean_tpr, color=color, lw=2.2,
             label=f"{CLASS_NAMES[c]} (AUC = {mean_auc:.3f} ± {std_auc:.3f})")

# Macro-average curve
macro_tpr = np.mean([all_tprs[c] for c in range(NUM_CLASSES)], axis=0)
macro_auc = np.mean([all_aucs[c][0] for c in range(NUM_CLASSES)])
macro_std = np.mean([all_aucs[c][1] for c in range(NUM_CLASSES)])

plt.plot(mean_fpr, macro_tpr, color="black", linestyle="--", lw=2.5,
         label=f"Macro-average (AUC = {macro_auc:.3f} ± {macro_std:.3f})")

# Diagonal
plt.plot([0, 1], [0, 1], linestyle=":", color="gray", lw=1.5)

plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.05])
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curves (5-Fold Mean ± Std)", fontsize=14, fontweight="bold")
plt.legend(loc="lower right", fontsize=9, framealpha=0.95)
plt.grid(True, alpha=0.3, linestyle="--")
plt.tight_layout()

save_path = os.path.join(RESULTS_DIR, "roc_curves_5fold.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"\nROC curve saved to: {save_path}")
print(f"Macro AUC: {macro_auc:.4f} ± {macro_std:.4f}")

In [ ]:
# ============================================================
# Confusion Matrix (Per-Fold + Aggregated) - Paper Ready
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
RESULTS_DIR = "/kaggle/working/kfold_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

N_FOLDS = 5
CLASS_NAMES = ["F0", "F1", "F2", "F3", "F4"]  
NUM_CLASSES = len(CLASS_NAMES)

# ============================================================
# 1. Aggregated Confusion Matrix (Raw Count)
# ============================================================
agg_cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)

for fold in range(1, N_FOLDS + 1):
    cm_path = os.path.join(CHECKPOINT_DIR, f"fold_{fold}", "confusion_matrix.npy")
    
    if os.path.exists(cm_path):
        cm = np.load(cm_path)
        agg_cm += cm
        print(f"Fold {fold} CM loaded")
    else:
        
        labels_path = os.path.join(CHECKPOINT_DIR, f"fold_{fold}", "test_labels.npy")
        probs_path  = os.path.join(CHECKPOINT_DIR, f"fold_{fold}", "test_probs.npy")
        
        if os.path.exists(labels_path) and os.path.exists(probs_path):
            labels = np.load(labels_path)
            preds = np.load(probs_path).argmax(axis=1)
            cm = confusion_matrix(labels, preds, labels=range(NUM_CLASSES))
            agg_cm += cm
            print(f"Fold {fold} CM computed from probs/labels")
        else:
            print(f"Fold {fold}: CM data missing!")

print("\nAggregated Confusion Matrix (Counts):")
print(agg_cm)

# ============================================================
# 2. Plot Aggregated Confusion Matrix (Count)
# ============================================================
plt.figure(figsize=(7, 6))
sns.heatmap(agg_cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, linecolor="white",
            annot_kws={"size": 13, "weight": "bold"})
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.title("Aggregated Confusion Matrix (5-Fold)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "aggregated_confusion_matrix_counts.png"), dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# 3. Normalized Confusion Matrix (Row-wise % ) - 
# ============================================================
agg_cm_norm = agg_cm.astype("float") / agg_cm.sum(axis=1)[:, np.newaxis]
agg_cm_norm = np.nan_to_num(agg_cm_norm)  # 0 division handle

plt.figure(figsize=(7, 6))
sns.heatmap(agg_cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, linecolor="white",
            annot_kws={"size": 13, "weight": "bold"},
            vmin=0, vmax=1)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.title("Normalized Confusion Matrix (5-Fold)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "aggregated_confusion_matrix_normalized.png"), dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# 4. Per-Fold Confusion Matrices 
# ============================================================
fig, axes = plt.subplots(1, N_FOLDS, figsize=(4*N_FOLDS, 4))

for fold in range(1, N_FOLDS + 1):
    ax = axes[fold-1]
    cm_path = os.path.join(CHECKPOINT_DIR, f"fold_{fold}", "confusion_matrix.npy")
    
    if os.path.exists(cm_path):
        cm = np.load(cm_path)
    else:
        labels = np.load(os.path.join(CHECKPOINT_DIR, f"fold_{fold}", "test_labels.npy"))
        preds = np.load(os.path.join(CHECKPOINT_DIR, f"fold_{fold}", "test_probs.npy")).argmax(axis=1)
        cm = confusion_matrix(labels, preds, labels=range(NUM_CLASSES))
    
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                cbar=False, annot_kws={"size": 10})
    ax.set_title(f"Fold {fold}", fontsize=12)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True" if fold == 1 else "")

plt.suptitle("Per-Fold Confusion Matrices", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "per_fold_confusion_matrices.png"), dpi=300, bbox_inches="tight")
plt.show()

print("\nAll confusion matrices saved in:", RESULTS_DIR)

In [ ]:
# ============================================================
# Full Corrected Code:
# Ordinal Deviation Histogram + UMAP + Silhouette & Davies-Bouldin
# ============================================================

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import models as tv_models          # ← fixed import
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import pandas as pd

# -------------------- Config --------------------
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
RESULTS_DIR = "/kaggle/working/kfold_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_FOLDS = 5
NUM_CLASSES = 5
CLASS_NAMES = ["F0", "F1", "F2", "F3", "F4"]
BATCH_SIZE = 32
SEED = 42

print(f"Device: {DEVICE}")

# ============================================================
# StudentModel (fixed)
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        y = torch.cat([avg_out, max_out], dim=1)
        y = self.conv(y)
        return self.sigmoid(y)

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)
    def forward(self, x):
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x

class StudentModel(nn.Module):
    def __init__(self, num_classes, use_cbam=True):
        super().__init__()
        backbone = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.features = backbone.features
        out_channels = 1280
        self.use_cbam = use_cbam
        if use_cbam:
            self.attention = CBAM(out_channels)
        else:
            self.attention = SEBlock(out_channels)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(out_channels, num_classes)

    def forward(self, x, return_features=False):
        x = self.features(x)
        x = self.attention(x)
        x = self.pool(x)
        feat = torch.flatten(x, 1)
        out = self.classifier(self.dropout(feat))
        if return_features:
            return out, feat
        return out

# ============================================================
# 1. Extract predictions + features from all folds
# ============================================================
print("Extracting predictions and features...")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_indices = np.arange(len(targets))

all_preds = []
all_labels = []
all_features = []

for fold_idx, (trainval_idx, test_idx) in enumerate(skf.split(all_indices, targets)):
    fold_num = fold_idx + 1
    print(f"  Fold {fold_num}...")

    test_dataset = Subset(base_dataset_eval, test_idx)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    student = StudentModel(NUM_CLASSES, use_cbam=True).to(DEVICE)
    student_path = os.path.join(CHECKPOINT_DIR, f"fold_{fold_num}", "student_model.pth")
    student.load_state_dict(torch.load(student_path, map_location=DEVICE))
    student.eval()

    fold_feats, fold_preds, fold_labs = [], [], []

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE)
            logits, feats = student(x, return_features=True)
            probs = F.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)

            fold_feats.append(feats.cpu().numpy())
            fold_preds.append(preds.cpu().numpy())
            fold_labs.append(y.numpy())

    all_features.append(np.concatenate(fold_feats))
    all_preds.append(np.concatenate(fold_preds))
    all_labels.append(np.concatenate(fold_labs))

# Concatenate all folds
features = np.concatenate(all_features)
preds    = np.concatenate(all_preds)
labels   = np.concatenate(all_labels)

print(f"\nTotal samples collected: {len(labels)}")

# ============================================================
# 2. Ordinal Deviation Histogram
# ============================================================
deviation = preds - labels

plt.figure(figsize=(6.5, 4.8))
bins = np.arange(-4.5, 5.5, 1)
plt.hist(deviation, bins=bins, color="#E67E22", edgecolor="black", linewidth=0.8)
plt.xlabel("Ordinal Deviation (Predicted − True)", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.title("(a) Ordinal Deviation Histogram", fontsize=13, fontweight="bold")
plt.xticks(range(-4, 5))
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "ordinal_deviation_histogram.png"), dpi=300, bbox_inches="tight")
plt.show()

print("\nOrdinal Deviation Distribution:")
unique, counts = np.unique(deviation, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Deviation {u:+d}: {c:4d} samples ({c/len(deviation)*100:.1f}%)")

# ============================================================
# 3. UMAP Embedding
# ============================================================
print("\nRunning UMAP (this may take a minute)...")
scaler = StandardScaler()
feats_scaled = scaler.fit_transform(features)

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=SEED, verbose=False)
embedding = reducer.fit_transform(feats_scaled)

plt.figure(figsize=(7.5, 6))
palette = sns.color_palette("Spectral", NUM_CLASSES)
for c in range(NUM_CLASSES):
    mask = labels == c
    plt.scatter(embedding[mask, 0], embedding[mask, 1],
                c=[palette[c]], label=CLASS_NAMES[c],
                s=22, alpha=0.75, edgecolors="none")

plt.xlabel("UMAP-1", fontsize=12)
plt.ylabel("UMAP-2", fontsize=12)
plt.title("(b) UMAP Embedding of Student Features", fontsize=13, fontweight="bold")
plt.legend(title="Fibrosis Stage", fontsize=10, markerscale=1.6, framealpha=0.95)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "umap_embedding.png"), dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# 4. Silhouette Score & Davies-Bouldin Index (per fold)
# ============================================================
print("\nComputing Silhouette & Davies-Bouldin Index...")

sil_scores = []
db_scores = []

for fold in range(N_FOLDS):
    f = all_features[fold]
    l = all_labels[fold]

    f_scaled = StandardScaler().fit_transform(f)

    sil = silhouette_score(f_scaled, l)
    db  = davies_bouldin_score(f_scaled, l)

    sil_scores.append(sil)
    db_scores.append(db)
    print(f"  Fold {fold+1}:  Silhouette = {sil:.3f}  |  DB Index = {db:.3f}")

print("\n" + "="*50)
print(f"Mean Silhouette ↑ : {np.mean(sil_scores):.3f} ± {np.std(sil_scores):.3f}")
print(f"Mean DB Index   ↓ : {np.mean(db_scores):.3f} ± {np.std(db_scores):.3f}")
print("="*50)

# Create nice table
df = pd.DataFrame({
    "Fold": list(range(1, N_FOLDS+1)) + ["Mean ± Std"],
    "Silhouette ↑": [f"{s:.3f}" for s in sil_scores] + [f"{np.mean(sil_scores):.3f} ± {np.std(sil_scores):.3f}"],
    "DB index ↓"  : [f"{d:.3f}" for d in db_scores] + [f"{np.mean(db_scores):.3f} ± {np.std(db_scores):.3f}"]
})

print("\n" + df.to_string(index=False))
df.to_csv(os.path.join(RESULTS_DIR, "latent_space_metrics.csv"), index=False)

print("\nAll files saved to:", RESULTS_DIR)
print("Done!")

In [ ]:
# ============================================================
# Updated UMAP Embedding with Clear Misclassification Highlight
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import umap
import os

RESULTS_DIR = "/kaggle/working/kfold_results"
CLASS_NAMES = ["F0", "F1", "F2", "F3", "F4"]
NUM_CLASSES = 5
SEED = 42



print("Creating improved UMAP with misclassifications...")

# ----- UMAP -----
scaler = StandardScaler()
feats_scaled = scaler.fit_transform(features)

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, 
                    random_state=SEED, verbose=False)
embedding = reducer.fit_transform(feats_scaled)

correct_mask = (preds == labels)
wrong_mask   = ~correct_mask

print(f"Correctly classified : {correct_mask.sum()}")
print(f"Misclassified        : {wrong_mask.sum()}")

# ============================================================
# Improved Plot
# ============================================================
plt.figure(figsize=(10, 8))


palette = sns.color_palette("Set2", NUM_CLASSES)  

# ----- 1. Correct predictions (soft dots) -----
for c in range(NUM_CLASSES):
    mask = (labels == c) & correct_mask
    plt.scatter(embedding[mask, 0], embedding[mask, 1],
                c=[palette[c]], 
                s=28, alpha=0.65, 
                edgecolors="none",
                label=f"{CLASS_NAMES[c]} (Correct)",
                zorder=2)

# ----- 2. Misclassified (big circle + red edge + black center) -----
for c in range(NUM_CLASSES):
    mask = (labels == c) & wrong_mask
    if mask.sum() > 0:
        plt.scatter(embedding[mask, 0], embedding[mask, 1],
                    c=[palette[c]], 
                    s=110, alpha=0.95,
                    edgecolors="red", linewidths=1.8,
                    marker="o",
                    zorder=5)


plt.scatter([], [], c="gray", s=110, edgecolors="red", linewidths=1.8,
            marker="o", label="Misclassified", alpha=0.95)

plt.xlabel("UMAP-1", fontsize=13)
plt.ylabel("UMAP-2", fontsize=13)
plt.title("UMAP Embedding of Student Features\n(Misclassifications Highlighted in Red Border)", 
          fontsize=14, fontweight="bold", pad=12)

plt.legend(title="Fibrosis Stage", fontsize=10, title_fontsize=11,
           markerscale=1.2, framealpha=0.95, loc="best")

plt.grid(True, alpha=0.25, linestyle="--")
plt.tight_layout()

save_path = os.path.join(RESULTS_DIR, "umap_misclassification_updated.png")
plt.savefig(save_path, dpi=600, bbox_inches="tight")
plt.show()

print(f"\nSaved to: {save_path}")